wheee architecture

In [32]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, ConcatDataset, random_split
from torch.nn.utils.rnn import pad_sequence
import time
import copy
import os
from pathlib import Path
import glob

In [36]:
class SleepDataset(Dataset):
    def __init__(self, data_dir):
        
        file_paths = glob.glob(os.path.join(data_dir, '*.pt'))
        self.file_paths = file_paths
        # self.data = torch.load(clean_dir / f"EPCTL01_dataset.pt",  weights_only=False)
    def __len__(self):
        return len(self.file_paths)
    def __getitem__(self, idx):
        file_path = self.file_paths[idx]
        data = torch.load(file_path, weights_only=False, mmap=True)
        feature_tensor = torch.from_numpy(data["X"]).float() # Cast to appropriate dtype
        label_tensor = torch.from_numpy(data["y"]).long() # Cast to appropriate dtype for labels
        return feature_tensor, label_tensor
data_dir = 'anphy_sleep_data/patient_records/clean'       
combined_dataset = SleepDataset(data_dir)


In [37]:
combined_dataset[0]

(tensor([[[ 4.6934e-05,  4.8292e-05,  4.3852e-05,  ..., -1.0177e-04,
           -9.4053e-05, -9.9274e-05],
          [ 1.3811e-05,  1.3724e-05,  1.5328e-05,  ..., -1.0412e-04,
           -1.0248e-04, -1.1347e-04],
          [ 1.3609e-06,  2.6023e-06,  1.7542e-06,  ..., -2.8776e-05,
           -3.6223e-05, -3.6634e-05],
          ...,
          [-2.7262e-05, -2.0372e-05, -1.5078e-05,  ...,  5.5894e-05,
            4.7794e-05,  6.3008e-05],
          [ 3.6070e-05,  4.3928e-05,  5.1304e-05,  ..., -2.4004e-05,
            4.0736e-05,  8.7968e-06],
          [ 2.8035e-05,  2.5980e-05,  2.5471e-05,  ...,  1.8516e-05,
            2.1854e-05,  1.7213e-05]],
 
         [[-1.0069e-04, -1.0315e-04, -1.0689e-04,  ...,  3.7398e-05,
            3.4358e-05,  4.4723e-05],
          [-1.1298e-04, -1.2108e-04, -1.1926e-04,  ..., -1.0063e-05,
           -1.5735e-05, -9.1422e-06],
          [-3.1266e-05, -4.1474e-05, -3.4080e-05,  ...,  1.3327e-05,
            9.8148e-06,  1.3894e-05],
          ...,
    

In [28]:
def custom_collate(batch):
    # batch is a list of tuples: [(data, label), ...]
    # Separate data and labels
    data_list, labels_list = zip(*batch) 
    # Pad sequences to the length of the longest in the batch
    padded_data = pad_sequence(data_list, batch_first=True)
    return padded_data, torch.tensor(labels_list)


In [4]:
class MyModel(nn.Module):
    # You can use pre-existing models but change layers to recieve full credit.
    def __init__(self):
        super(MyModel, self).__init__()
        ### TODO: BEGIN SOLUTION ###
        self.model_sequential = nn.Sequential(
            nn.Conv1d(in_channels=93, out_channels= 32, kernel_size=3),
            nn.BatchNorm1d(num_features=32),
            nn.ReLU(),
            nn.MaxPool1d(2, 2),
            nn.Dropout(0.2),
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Dropout(0.2),

            nn.Conv1d(in_channels=64, out_channels = 128, kernel_size=3),
            nn.BatchNorm1d(num_features=128),
            nn.ReLU(),
            nn.Conv1d(in_channels=128, out_channels = 128, kernel_size=3),
            nn.BatchNorm1d(num_features=128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Dropout(0.2),

            nn.Flatten(),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 10),
        )
        ### END SOLUTION ###

    def forward(self, x):
        outs = None
        ### TODO: BEGIN SOLUTION ###
        outs = self.model_sequential(x)
        ### END SOLUTION ###
        return outs

In [12]:
def get_device():
    if torch.cuda.is_available():
        return "cuda"
    elif torch.backends.mps.is_available():
        return "mps"
    else:
        return "cpu"

class AverageMeter(object):
    """Computes and stores the average and current value"""

    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def accuracy(output, target):
    """Computes the precision@k for the specified values of k"""
    batch_size = target.shape[0]

    _, pred = torch.max(output, dim=-1)

    correct = pred.eq(target).sum() * 1.0

    acc = correct.item() / batch_size

    return acc


def train(epoch, data_loader, model, optimizer, criterion):
    iter_time = AverageMeter()
    losses = AverageMeter()
    acc = AverageMeter()

    for idx, (data, target) in enumerate(data_loader):
        start = time.time()

        data = data.to(device)
        target = target.to(device)

        # calculate model predictions, training loss, and update model parameters
        ### TODO: BEGIN SOLUTION ###
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out, target)
        loss.backward()
        optimizer.step()
        
        ### END SOLUTION ###

        batch_acc = accuracy(out, target)

        losses.update(loss, out.shape[0])
        acc.update(batch_acc, out.shape[0])

        iter_time.update(time.time() - start)
        if idx % 10 == 0:
            print(
                (
                    "Epoch: [{0}][{1}/{2}]\t"
                    "Time {iter_time.val:.3f} ({iter_time.avg:.3f})\t"
                    "Loss {loss.val:.4f} ({loss.avg:.4f})\t"
                    "Prec @1 {top1.val:.4f} ({top1.avg:.4f})\t"
                ).format(
                    epoch,
                    idx,
                    len(data_loader),
                    iter_time=iter_time,
                    loss=losses,
                    top1=acc,
                )
            )


def validate(epoch, val_loader, model, criterion):
    """
    Hint: make sure to use torch.no_grad() to disable gradient computation. This will help reduce memory usage and speed up computation.
    """
    iter_time = AverageMeter()
    losses = AverageMeter()
    acc = AverageMeter()

    num_class = 6
    cm = torch.zeros(num_class, num_class)
    # evaluation loop
    for idx, (data, target) in enumerate(val_loader):
        start = time.time()

        data = data.to(device)
        target = target.to(device)

        # calculate model predictions and validation loss
        ### TODO: BEGIN SOLUTION ###
        with torch.no_grad():
            out = model(data)
            loss = criterion(out, target)

            
        ### END SOLUTION ###

        batch_acc = accuracy(out, target)

        # update confusion matrix
        _, preds = torch.max(out, 1)
        for t, p in zip(target.view(-1), preds.view(-1)):
            cm[t.long(), p.long()] += 1

        losses.update(loss, out.shape[0])
        acc.update(batch_acc, out.shape[0])

        iter_time.update(time.time() - start)
        if idx % 1 == 0:
            print(
                (
                    "Epoch: [{0}][{1}/{2}]\t"
                    "Time {iter_time.val:.3f} ({iter_time.avg:.3f})\t"
                ).format(
                    epoch,
                    idx,
                    len(val_loader),
                    iter_time=iter_time,
                    loss=losses,
                    top1=acc,
                )
            )
    cm = cm / cm.sum(1)
    per_cls_acc = cm.diag().detach().numpy().tolist()
    for i, acc_i in enumerate(per_cls_acc):
        print("Accuracy of Class {}: {:.4f}".format(i, acc_i))

    print("* Prec @1: {top1.avg:.4f}".format(top1=acc))
    return acc.avg, cm


def adjust_learning_rate(optimizer, learning_rate, epoch, warmup, steps):
    epoch += 1
    if epoch <= warmup:
        lr = learning_rate * epoch / warmup
    elif epoch > steps[1]:
        lr = learning_rate * 0.01
    elif epoch > steps[0]:
        lr = learning_rate * 0.1
    else:
        lr = learning_rate
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr


In [13]:
def main(lr, momentum, weight_decay, epochs, warmup, steps):
    device = get_device()
    
    model = MyModel()
    
    train_size = int(len(combined_dataset) * 0.8)
    test_size = len(combined_dataset) - train_size
    train_dataset, test_dataset = random_split(combined_dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr,
        momentum,
        weight_decay
    )
        
    best = 0.0
    best_cm = None
    best_model = None
    for epoch in range(epochs):
        adjust_learning_rate(optimizer, lr, epoch, warmup, steps)

        # train loop
        train(epoch, train_loader, model, optimizer, criterion)

        # validation loop
        acc, cm = validate(epoch, test_loader, model, criterion)

        if acc > best:
            best = acc
            best_cm = cm
            best_model = copy.deepcopy(model)

    print("Best Prec @1 Acccuracy: {:.4f}".format(best))
    per_cls_acc = best_cm.diag().detach().numpy().tolist()
    for i, acc_i in enumerate(per_cls_acc):
        print("Accuracy of Class {}: {:.4f}".format(i, acc_i))
    
    if not os.path.exists("./checkpoints"):
        os.makedirs("./checkpoints")
        torch.save(
            best_model.state_dict(), "./checkpoints/" + "best_model" + ".pth"
        )


In [14]:
main(lr = 0.01, momentum = 0.9, weight_decay = 0.001, epochs = 1, warmup = 0, steps = [6, 8])

KeyboardInterrupt: 

In [38]:
acc = AverageMeter()
device = get_device()
    
model = MyModel()

train_size = int(len(combined_dataset) * 0.8)
test_size = len(combined_dataset) - train_size
train_dataset, test_dataset = random_split(combined_dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))

In [39]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=12, persistent_workers=True, pin_memory=True, collate_fn=custom_collate)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=12, persistent_workers=True, pin_memory=True, collate_fn=custom_collate)

In [40]:
for data, target in test_loader:
    print("started loading")
    data = data.to(device)
    target = target.to(device)
    out = model(data)
    batch_acc = accuracy(out, target)
    acc.update(batch_acc, out.shape[0])
    break
self.assertGreater(acc.avg, 0.4)

TypeError: Caught TypeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/hice1/eyang82/.conda/envs/anphy_sleep_env/lib/python3.11/site-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/hice1/eyang82/.conda/envs/anphy_sleep_env/lib/python3.11/site-packages/torch/utils/data/_utils/fetch.py", line 57, in fetch
    return self.collate_fn(data)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_342577/1951759802.py", line 7, in custom_collate
    return padded_data, torch.tensor(labels_list)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: only integer tensors of a single element can be converted to an index
